# 07 LANDFIRE Vegetation + Fuel Enrichment

This notebook adds **LANDFIRE vegetation and fuel features** to the current wildfire modeling table.

Preferred input:

```text
data/processed/calfire_with_gridmet_terrain_drought.csv
```

Fallback input:

```text
data/processed/calfire_with_gridmet_terrain.csv
```

Outputs:

```text
data/raw/landfire/
data/interim/landfire/
data/processed/landfire_vegetation_fuel_features.csv
data/processed/calfire_with_gridmet_terrain_drought_veg.csv
```

Core workflow:

```text
fire latitude/longitude points
→ create LANDFIRE AOI
→ request LANDFIRE fuel/vegetation layers through LFPS
→ download multiband GeoTIFF
→ sample each band at each fire point
→ engineer broad fuel model group
→ merge vegetation/fuel features into modeling table
```

Important modeling note:

This notebook keeps high-cardinality vegetation layers like `evt_code` and `fvt_code` as raw sampled codes. Do **not** one-hot encode every raw vegetation type yet. Start modeling with broader features like `fbfm40_group`, cover/height codes, and canopy cover.


## 0. Package setup

Install once if needed:

```powershell
pip install pandas numpy geopandas rasterio requests tqdm shapely
```

If Windows has issues with geospatial packages, use conda/mamba:

```powershell
conda install -c conda-forge pandas numpy geopandas rasterio requests tqdm shapely
```


In [1]:
from pathlib import Path
import json
import time
import zipfile
import re
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from shapely.geometry import box
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_LANDFIRE_DIR = RAW_DIR / "landfire"
INTERIM_LANDFIRE_DIR = INTERIM_DIR / "landfire"

for path in [RAW_LANDFIRE_DIR, INTERIM_LANDFIRE_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

PREFERRED_INPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain_drought.csv"
FALLBACK_INPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain.csv"

if PREFERRED_INPUT_PATH.exists():
    INPUT_PATH = PREFERRED_INPUT_PATH
else:
    INPUT_PATH = FALLBACK_INPUT_PATH

VEG_FEATURES_PATH = PROCESSED_DIR / "landfire_vegetation_fuel_features.csv"
OUTPUT_PATH = PROCESSED_DIR / "calfire_with_gridmet_terrain_drought_veg.csv"

LFPS_TASK_URL = "https://lfps.usgs.gov/arcgis/rest/services/LandfireProductService/GPServer/LandfireProductService"
LFPS_SUBMIT_URL = f"{LFPS_TASK_URL}/submitJob"
LFPS_PRODUCT_TABLE_URL = "https://lfps.usgs.gov/products"

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("Raw LANDFIRE folder:", RAW_LANDFIRE_DIR)
print("Vegetation/fuel features output:", VEG_FEATURES_PATH)
print("Merged output:", OUTPUT_PATH)


Project root: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2
Input path: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain_drought.csv
Raw LANDFIRE folder: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\landfire
Vegetation/fuel features output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\landfire_vegetation_fuel_features.csv
Merged output: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain_drought_veg.csv


c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load current modeling table

This notebook uses the drought-enriched table if available. Otherwise, it falls back to the terrain-only table.


In [2]:
df = pd.read_csv(INPUT_PATH)

required_cols = ["gridmet_id", "Latitude", "Longitude"]
missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["gridmet_id"] = df["gridmet_id"].astype(str)
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")

pre_drop = len(df)
df = df.dropna(subset=["gridmet_id", "Latitude", "Longitude"]).copy()
df = df[df["Latitude"].between(32, 42)].copy()
df = df[df["Longitude"].between(-125, -113)].copy()

print("Rows loaded:", pre_drop)
print("Rows after coordinate filter:", len(df))
display(df[["gridmet_id", "Name", "Latitude", "Longitude", "AcresBurned", "severity_tier"]].head())


Rows loaded: 2397
Rows after coordinate filter: 2397


,gridmet_id,Name,Latitude,Longitude,AcresBurned,severity_tier
0,fire_00000,Creek Fire,38.409580,-122.431720,65.0,Small
1,fire_00001,Taglio Fire,37.217100,-121.080360,30.0,Small
2,fire_00002,Tulloch Fire,37.927613,-120.528836,85.0,Small
3,fire_00003,Metz Fire,36.381230,-121.200590,3876.0,Large
4,fire_00004,Wheatland Fire,34.276000,-118.354000,156.0,Medium


## 2. Convert fires to geospatial points and create AOI

The LANDFIRE Product Service requires an AOI as:

```text
west south east north
```

in WGS84 longitude/latitude coordinates.


In [3]:
fires_gdf = gpd.GeoDataFrame(
    df[["gridmet_id", "Name", "Latitude", "Longitude"]].copy(),
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)

buffer_degrees = 0.25
minx, miny, maxx, maxy = fires_gdf.total_bounds

aoi_bounds = (
    minx - buffer_degrees,
    miny - buffer_degrees,
    maxx + buffer_degrees,
    maxy + buffer_degrees,
)

aoi_wsen = f"{aoi_bounds[0]:.6f} {aoi_bounds[1]:.6f} {aoi_bounds[2]:.6f} {aoi_bounds[3]:.6f}"

aoi_gdf = gpd.GeoDataFrame(
    {"name": ["fire_points_landfire_aoi"]},
    geometry=[box(*aoi_bounds)],
    crs="EPSG:4326",
)

aoi_path = INTERIM_LANDFIRE_DIR / "fire_points_landfire_aoi.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")

print("AOI west south east north:")
print(aoi_wsen)
print("Saved AOI:", aoi_path)
display(aoi_gdf)


AOI west south east north:
-124.612017 32.307546 -114.026308 42.244830
Saved AOI: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\interim\landfire\fire_points_landfire_aoi.geojson


,name,geometry
0,fire_points_landfire_aoi,"POLYGON ((-114.02631 32.30755, -114.02631 42.2..."


## 3. LANDFIRE layer selection

Recommended first-pass layers:

| Feature | LANDFIRE product |
|---|---|
| `fbfm40_code` | 40 Scott and Burgan Fire Behavior Fuel Models |
| `evt_code` | Existing Vegetation Type |
| `evc_code` | Existing Vegetation Cover |
| `evh_code` | Existing Vegetation Height |
| `canopy_cover` | Forest Canopy Cover |
| `fvt_code` | Fuel Vegetation Type |
| `fvc_code` | Fuel Vegetation Cover |

LANDFIRE has been updating naming conventions. This notebook starts with the current-style LF2024 layer names, with older-style fallback names available if needed.


In [4]:
# Current-style LFPS layer names based on LANDFIRE's updated naming convention.
CURRENT_LAYER_MAP = {
    "fbfm40_code": "LF2024_FBFM40",
    "evt_code": "LF2024_EVT",
    "evc_code": "LF2024_EVC",
    "evh_code": "LF2024_EVH",
    "canopy_cover": "LF2024_CC",
    "fvt_code": "LF2024_FVT",
    "fvc_code": "LF2024_FVC",
}

# Older-style fallback names. Use only if the current names fail.
OLD_LAYER_MAP = {
    "fbfm40_code": "240FBFM40",
    "evt_code": "250EVT",
    "evc_code": "250EVC",
    "evh_code": "250EVH",
    "canopy_cover": "240CC",
    "fvt_code": "240FVT",
    "fvc_code": "240FVC",
}

# Start with current names.
LANDFIRE_FEATURE_TO_LAYER = CURRENT_LAYER_MAP.copy()

LANDFIRE_FEATURE_NAMES = list(LANDFIRE_FEATURE_TO_LAYER.keys())
LANDFIRE_LAYER_NAMES = list(LANDFIRE_FEATURE_TO_LAYER.values())
LANDFIRE_LAYER_LIST = ";".join(LANDFIRE_LAYER_NAMES)

print("Requested feature names:")
print(LANDFIRE_FEATURE_NAMES)
print("\nRequested LFPS layer names:")
print(LANDFIRE_LAYER_NAMES)
print("\nLayer_List parameter:")
print(LANDFIRE_LAYER_LIST)

layer_metadata = pd.DataFrame({
    "feature_name": LANDFIRE_FEATURE_NAMES,
    "lfps_layer_name": LANDFIRE_LAYER_NAMES,
})
layer_metadata.to_csv(INTERIM_LANDFIRE_DIR / "landfire_requested_layers.csv", index=False)
display(layer_metadata)


Requested feature names:
['fbfm40_code', 'evt_code', 'evc_code', 'evh_code', 'canopy_cover', 'fvt_code', 'fvc_code']

Requested LFPS layer names:
['LF2024_FBFM40', 'LF2024_EVT', 'LF2024_EVC', 'LF2024_EVH', 'LF2024_CC', 'LF2024_FVT', 'LF2024_FVC']

Layer_List parameter:
LF2024_FBFM40;LF2024_EVT;LF2024_EVC;LF2024_EVH;LF2024_CC;LF2024_FVT;LF2024_FVC


,feature_name,lfps_layer_name
0,fbfm40_code,LF2024_FBFM40
1,evt_code,LF2024_EVT
2,evc_code,LF2024_EVC
3,evh_code,LF2024_EVH
4,canopy_cover,LF2024_CC
5,fvt_code,LF2024_FVT
6,fvc_code,LF2024_FVC


## 4. Optional: check the LFPS product table

If this cell fails, it is not fatal. It just helps verify whether the layer names appear on the product table.


In [5]:
def fetch_lfps_product_table():
    try:
        r = requests.get(LFPS_PRODUCT_TABLE_URL, timeout=60)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print("Could not fetch LFPS product table:", e)
        return None


product_table_text = fetch_lfps_product_table()

if product_table_text:
    lower_text = product_table_text.lower()
    for layer in LANDFIRE_LAYER_NAMES:
        found = layer.lower() in lower_text
        print(f"{layer:20s} found_in_product_table={found}")

    (INTERIM_LANDFIRE_DIR / "lfps_product_table_raw.html").write_text(product_table_text, encoding="utf-8")
else:
    print("Skipping product table verification.")


LF2024_FBFM40        found_in_product_table=False
LF2024_EVT           found_in_product_table=False
LF2024_EVC           found_in_product_table=False
LF2024_EVH           found_in_product_table=False
LF2024_CC            found_in_product_table=False
LF2024_FVT           found_in_product_table=False
LF2024_FVC           found_in_product_table=False


## 5. Submit LANDFIRE Product Service job

The previous version used the ArcGIS `submitJob` endpoint directly. That endpoint can return an HTML form instead of JSON, which caused the `JSONDecodeError`.

This version uses the newer LFPS API endpoint:

```text
https://lfps.usgs.gov/api/job/submit
```

You must provide an email address. LFPS requires this for job submission/usage tracking.

Before running the next cell, replace:

```python
LANDFIRE_EMAIL = "your_email@example.com"
```

with your email address.

Default resolution:

```python
LANDFIRE_RESOLUTION_M = 500
```

If the AOI is too large, use:

```python
LANDFIRE_RESOLUTION_M = 1000
```


In [6]:
LANDFIRE_RESOLUTION_M = 500
OUTPUT_PROJECTION = 4326  # WKID for WGS84. Use integer, not "EPSG:4326".

# Required by the current LFPS API.
# Replace this before running.
LANDFIRE_EMAIL = "chau.devin031602@gmail.com"

LFPS_API_SUBMIT_URL = "https://lfps.usgs.gov/api/job/submit"
job_metadata_path = INTERIM_LANDFIRE_DIR / "landfire_lfps_job_metadata.json"


def submit_lfps_job_v2(
    layer_list,
    aoi_wsen,
    email,
    output_projection=4326,
    resolution_m=500,
):
    if not email or email == "your_email@example.com" or "@" not in email:
        raise ValueError(
            "Set LANDFIRE_EMAIL to a real email address before submitting the LFPS job."
        )

    params = {
        "Email": email,
        "Layer_List": layer_list,
        "Area_of_Interest": aoi_wsen,
        "Output_Projection": output_projection,
        "Resample_Resolution": resolution_m,
    }

    response = requests.get(
        LFPS_API_SUBMIT_URL,
        params=params,
        headers={"Accept": "application/json", "User-Agent": "wildfire-severity-v2"},
        timeout=120,
    )

    debug_response_path = INTERIM_LANDFIRE_DIR / "landfire_submit_raw_response.txt"
    debug_response_path.write_text(response.text[:10000], encoding="utf-8", errors="replace")

    try:
        payload = response.json()
    except Exception as exc:
        print("LFPS did not return JSON.")
        print("HTTP status:", response.status_code)
        print("Content-Type:", response.headers.get("Content-Type"))
        print("First 1000 response characters:")
        print(response.text[:1000])
        print("Raw response saved to:", debug_response_path)
        raise exc

    if response.status_code != 200:
        print(json.dumps(payload, indent=2)[:3000])
        raise RuntimeError(f"LFPS request failed with status {response.status_code}")

    if "jobId" not in payload:
        print("LFPS response:")
        print(json.dumps(payload, indent=2)[:3000])
        raise RuntimeError("LFPS response did not include jobId.")

    return payload, params, response.url


# Set SUBMIT_NEW_JOB = True when you are ready to submit.
SUBMIT_NEW_JOB = True

if SUBMIT_NEW_JOB:
    job, request_params, request_url = submit_lfps_job_v2(
        layer_list=LANDFIRE_LAYER_LIST,
        aoi_wsen=aoi_wsen,
        email=LANDFIRE_EMAIL,
        output_projection=OUTPUT_PROJECTION,
        resolution_m=LANDFIRE_RESOLUTION_M,
    )

    job_info = {
        "job": job,
        "job_id": job.get("jobId"),
        "request_url": request_url,
        "request_params": request_params,
        "layer_map": LANDFIRE_FEATURE_TO_LAYER,
        "layer_list": LANDFIRE_LAYER_LIST,
        "aoi_wsen": aoi_wsen,
        "resolution_m": LANDFIRE_RESOLUTION_M,
        "output_projection": OUTPUT_PROJECTION,
    }

    job_metadata_path.write_text(json.dumps(job_info, indent=2), encoding="utf-8")
    print("Submitted job.")
    print("Job ID:", job_info["job_id"])
    print("Saved job metadata:", job_metadata_path)
else:
    print("SUBMIT_NEW_JOB is False. Existing job metadata will be used if available.")


Submitted job.
Job ID: c84b6d78-ac4d-4fbc-8262-0d4dca2bf691
Saved job metadata: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\interim\landfire\landfire_lfps_job_metadata.json


## 6. Poll LFPS job status

This still uses the ArcGIS job-status endpoint because the LFPS API returns an ArcGIS-style job ID.

If this cell reports `esriJobFailed`, inspect the saved status JSON in:

```text
data/interim/landfire/
```

Common fixes:
1. Increase `LANDFIRE_RESOLUTION_M` to `1000`
2. Switch from current layer names to the older layer map
3. Reduce the AOI or use a manual LANDFIRE download


In [7]:
def get_job_id_from_metadata(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return data["job_id"]


LFPS_API_STATUS_URL = "https://lfps.usgs.gov/api/job/status"


def poll_lfps_job_v2(job_id, sleep_seconds=30, max_minutes=180):
    started = time.time()

    while True:
        response = requests.get(
            LFPS_API_STATUS_URL,
            params={"JobId": job_id},
            headers={"Accept": "application/json", "User-Agent": "wildfire-severity-v2"},
            timeout=120,
        )
        response.raise_for_status()
        status = response.json()

        status_path = INTERIM_LANDFIRE_DIR / f"landfire_job_status_{job_id}.json"
        status_path.write_text(json.dumps(status, indent=2), encoding="utf-8")

        job_status = status.get("status")
        queue_position = status.get("queuePosition")
        output_file = status.get("outputFile")
        messages = status.get("messages")

        print(time.strftime("%H:%M:%S"), "status:", job_status, "queue:", queue_position)

        if messages:
            print("messages:", messages[-2:] if isinstance(messages, list) else messages)

        if output_file:
            print("Output file ready:", output_file)

        if job_status and "Succeeded" in str(job_status):
            return status

        if job_status and "Failed" in str(job_status):
            print(json.dumps(status, indent=2)[:5000])
            raise RuntimeError("LFPS job failed.")

        if job_status and any(x in str(job_status) for x in ["Cancelled", "Timed out", "TimedOut"]):
            print(json.dumps(status, indent=2)[:5000])
            raise RuntimeError(f"LFPS job ended with status: {job_status}")

        if (time.time() - started) > max_minutes * 60:
            print(json.dumps(status, indent=2)[:5000])
            raise TimeoutError("LFPS job polling timed out.")

        time.sleep(sleep_seconds)


job_id = get_job_id_from_metadata(job_metadata_path)
status = poll_lfps_job_v2(job_id)

print("Final status:")
print(json.dumps(status, indent=2)[:5000])

02:39:54 status: Pending queue: 1
02:40:24 status: Executing queue: -1
messages: [{'type': 'esriJobMessageTypeInformative', 'description': 'Start creating geotif'}, {'type': 'esriJobMessageTypeInformative', 'description': 'Start resample of geotif'}]
02:40:55 status: Executing queue: -1
messages: [{'type': 'esriJobMessageTypeInformative', 'description': 'Finished resample of geotif'}, {'type': 'esriJobMessageTypeInformative', 'description': 'Finished creating geotif'}]
02:41:26 status: Succeeded queue: -1
messages: [{'type': 'esriJobMessageTypeInformative', 'description': 'Job Finished'}, {'type': 'esriJobMessageTypeInformative', 'description': 'Succeeded at Friday, May 8, 2026 4:41:01 AM (Elapsed Time: 1 minutes 6 seconds)'}]
Output file ready: https://lfps.usgs.gov/arcgis/rest/directories/arcgisjobs/landfireproductservice_gpserver/j57919e6b53e142f78e7ae801837ec006/scratch/j57919e6b53e142f78e7ae801837ec006.zip
Final status:
{
  "jobId": "c84b6d78-ac4d-4fbc-8262-0d4dca2bf691",
  "queue

## 7. Download LFPS output ZIP

The result endpoint usually exposes the output file URL. This cell tries the result endpoint first, then a known ArcGIS scratch ZIP fallback.


In [8]:
def download_file(url, out_path, timeout=300):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with out_path.open("wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return out_path


job_id = get_job_id_from_metadata(job_metadata_path)

status_path = INTERIM_LANDFIRE_DIR / f"landfire_job_status_{job_id}.json"
status = json.loads(status_path.read_text(encoding="utf-8"))

output_url = status.get("outputFile")

if not output_url:
    print(json.dumps(status, indent=2)[:5000])
    raise ValueError("No outputFile found in LFPS status response.")

print("Output URL:", output_url)

zip_path = RAW_LANDFIRE_DIR / f"landfire_lfps_{job_id}.zip"

if not zip_path.exists():
    download_file(output_url, zip_path)
else:
    print("ZIP already exists:", zip_path)

print("ZIP path:", zip_path)
print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

Output URL: https://lfps.usgs.gov/arcgis/rest/directories/arcgisjobs/landfireproductservice_gpserver/j57919e6b53e142f78e7ae801837ec006/scratch/j57919e6b53e142f78e7ae801837ec006.zip
ZIP path: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\landfire\landfire_lfps_c84b6d78-ac4d-4fbc-8262-0d4dca2bf691.zip
Size MB: 8.56


## 8. Extract ZIP and find GeoTIFF

The LFPS output should contain a multiband GeoTIFF. The band order should match the requested layer order.


In [9]:
job_id = get_job_id_from_metadata(job_metadata_path)
zip_path = RAW_LANDFIRE_DIR / f"landfire_lfps_{job_id}.zip"
extract_dir = RAW_LANDFIRE_DIR / f"landfire_lfps_{job_id}"

if not extract_dir.exists():
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
else:
    print("Extract directory already exists:", extract_dir)

tif_files = sorted(list(extract_dir.rglob("*.tif")) + list(extract_dir.rglob("*.tiff")))

print("GeoTIFF files found:", len(tif_files))
for p in tif_files[:20]:
    print(p)

if not tif_files:
    raise FileNotFoundError("No GeoTIFF found in LFPS output ZIP.")

# Pick the largest GeoTIFF, which should usually be the product file.
LANDFIRE_RASTER_PATH = max(tif_files, key=lambda p: p.stat().st_size)
print("\nSelected raster:")
print(LANDFIRE_RASTER_PATH)
print("Size MB:", round(LANDFIRE_RASTER_PATH.stat().st_size / 1024 / 1024, 2))


GeoTIFF files found: 1
c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\landfire\landfire_lfps_c84b6d78-ac4d-4fbc-8262-0d4dca2bf691\j57919e6b53e142f78e7ae801837ec006.tif

Selected raster:
c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\landfire\landfire_lfps_c84b6d78-ac4d-4fbc-8262-0d4dca2bf691\j57919e6b53e142f78e7ae801837ec006.tif
Size MB: 8.55


## 9. Manual fallback option

If LFPS automation fails, manually download a LANDFIRE multiband GeoTIFF through the LANDFIRE Map Viewer or LFPS form, then set:

```python
LANDFIRE_RASTER_PATH = RAW_LANDFIRE_DIR / "your_file.tif"
```

Make sure the band order matches `LANDFIRE_FEATURE_NAMES`.


In [10]:
# Uncomment and edit this only if you manually downloaded a raster.
# LANDFIRE_RASTER_PATH = RAW_LANDFIRE_DIR / "manual_landfire_vegetation_fuels.tif"

print("Current LANDFIRE_RASTER_PATH:")
print(LANDFIRE_RASTER_PATH)


Current LANDFIRE_RASTER_PATH:
c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\raw\landfire\landfire_lfps_c84b6d78-ac4d-4fbc-8262-0d4dca2bf691\j57919e6b53e142f78e7ae801837ec006.tif


## 10. Inspect LANDFIRE raster

This confirms the CRS, band count, and raster size before sampling.


In [11]:
with rasterio.open(LANDFIRE_RASTER_PATH) as src:
    print("CRS:", src.crs)
    print("Band count:", src.count)
    print("Width x Height:", src.width, "x", src.height)
    print("Nodata:", src.nodata)
    print("Bounds:", src.bounds)

    if src.count != len(LANDFIRE_FEATURE_NAMES):
        print("\nWARNING: Band count does not match requested feature count.")
        print("Band count:", src.count)
        print("Expected:", len(LANDFIRE_FEATURE_NAMES))
        print("You may need to adjust LANDFIRE_FEATURE_NAMES to match the raster band order.")


CRS: EPSG:4326
Band count: 7
Width x Height: 1721 x 1634
Nodata: -9999.0
Bounds: BoundingBox(left=-124.62303817151121, bottom=32.303152502610736, right=-114.01807896318033, top=42.37200976374533)


## 11. Sample LANDFIRE raster at fire points

This samples each raster band at each wildfire ignition point.


In [12]:
def sample_multiband_raster_at_points(raster_path, points_gdf, band_names):
    raster_path = Path(raster_path)

    with rasterio.open(raster_path) as src:
        if len(band_names) != src.count:
            raise ValueError(
                f"band_names length ({len(band_names)}) does not match raster band count ({src.count})."
            )

        points_projected = points_gdf.to_crs(src.crs)
        coords = [(geom.x, geom.y) for geom in points_projected.geometry]

        samples = np.array(list(src.sample(coords)))

        out = pd.DataFrame(samples, columns=band_names)
        out.insert(0, "gridmet_id", points_gdf["gridmet_id"].astype(str).values)

        nodata_values = {-9999, -999, 32767, 65535, 255}
        if src.nodata is not None:
            nodata_values.add(src.nodata)

        for col in band_names:
            out[col] = pd.to_numeric(out[col], errors="coerce")
            out.loc[out[col].isin(nodata_values), col] = np.nan

    return out


veg_samples = sample_multiband_raster_at_points(
    LANDFIRE_RASTER_PATH,
    fires_gdf,
    LANDFIRE_FEATURE_NAMES,
)

print("Vegetation/fuel samples:", veg_samples.shape)
display(veg_samples.head())

veg_samples.to_csv(INTERIM_LANDFIRE_DIR / "landfire_raw_point_samples.csv", index=False)


Vegetation/fuel samples: (2397, 8)


,gridmet_id,fbfm40_code,evt_code,evc_code,evh_code,canopy_cover,fvt_code,fvc_code
0,fire_00000,183.0,7901.0,14.0,14.0,45.0,2914.0,14.0
1,fire_00001,93.0,7984.0,64.0,64.0,0.0,2964.0,64.0
2,fire_00002,102.0,9301.0,351.0,304.0,0.0,2270.0,125.0
3,fire_00003,102.0,9301.0,342.0,303.0,0.0,2184.0,123.0
4,fire_00004,122.0,7092.0,224.0,215.0,0.0,2092.0,112.0


## 12. Engineer modeling-friendly fuel/vegetation features

The most useful engineered feature here is `fbfm40_group`, which collapses 40 fire behavior fuel models into broad groups.

Keep the raw `evt_code` and `fvt_code`, but do not one-hot encode all categories yet. They can be high-cardinality.


In [13]:
def fbfm40_group(code):
    if pd.isna(code):
        return np.nan

    try:
        code = int(code)
    except Exception:
        return np.nan

    # Common Scott-Burgan FBFM40 groups.
    if code in [91, 92, 93, 98, 99]:
        return "nonburnable_or_other"
    elif 101 <= code <= 109:
        return "grass"
    elif 121 <= code <= 124:
        return "grass_shrub"
    elif 141 <= code <= 149:
        return "shrub"
    elif 161 <= code <= 165:
        return "timber_understory"
    elif 181 <= code <= 189:
        return "timber_litter"
    elif 201 <= code <= 204:
        return "slash_blowdown"
    else:
        return "other"


veg_features = veg_samples.copy()

if "fbfm40_code" in veg_features.columns:
    veg_features["fbfm40_group"] = veg_features["fbfm40_code"].apply(fbfm40_group)

# Make these categorical codes explicit as nullable integer columns where possible.
categorical_code_cols = [
    "fbfm40_code",
    "evt_code",
    "evc_code",
    "evh_code",
    "fvt_code",
    "fvc_code",
]

for col in categorical_code_cols:
    if col in veg_features.columns:
        veg_features[col] = pd.to_numeric(veg_features[col], errors="coerce").round()

# Canopy cover should be numeric if present.
if "canopy_cover" in veg_features.columns:
    veg_features["canopy_cover"] = pd.to_numeric(veg_features["canopy_cover"], errors="coerce")

veg_features.to_csv(VEG_FEATURES_PATH, index=False)

print("Saved vegetation/fuel features:", VEG_FEATURES_PATH)
print("Shape:", veg_features.shape)
display(veg_features.head())

if "fbfm40_group" in veg_features.columns:
    print("FBFM40 group counts:")
    display(veg_features["fbfm40_group"].value_counts(dropna=False))


Saved vegetation/fuel features: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\landfire_vegetation_fuel_features.csv
Shape: (2397, 9)


,gridmet_id,fbfm40_code,evt_code,evc_code,evh_code,canopy_cover,fvt_code,fvc_code,fbfm40_group
0,fire_00000,183.0,7901.0,14.0,14.0,45.0,2914.0,14.0,timber_litter
1,fire_00001,93.0,7984.0,64.0,64.0,0.0,2964.0,64.0,nonburnable_or_other
2,fire_00002,102.0,9301.0,351.0,304.0,0.0,2270.0,125.0,grass
3,fire_00003,102.0,9301.0,342.0,303.0,0.0,2184.0,123.0,grass
4,fire_00004,122.0,7092.0,224.0,215.0,0.0,2092.0,112.0,grass_shrub


FBFM40 group counts:


fbfm40_group
grass                   731
grass_shrub             588
nonburnable_or_other    442
shrub                   333
timber_understory       164
timber_litter           139
Name: count, dtype: int64

## 13. Merge vegetation/fuel features into modeling table

In [14]:
base = pd.read_csv(INPUT_PATH)
base["gridmet_id"] = base["gridmet_id"].astype(str)

veg_features = pd.read_csv(VEG_FEATURES_PATH)
veg_features["gridmet_id"] = veg_features["gridmet_id"].astype(str)

merged = base.merge(veg_features, on="gridmet_id", how="left")

merged.to_csv(OUTPUT_PATH, index=False)

print("Base:", base.shape)
print("Vegetation/fuel features:", veg_features.shape)
print("Merged:", merged.shape)
print("Saved:", OUTPUT_PATH)

display(merged[[
    "gridmet_id",
    "Name",
    "AcresBurned",
    "severity_tier",
    "fbfm40_code",
    "fbfm40_group",
    "evc_code",
    "evh_code",
    "canopy_cover",
]].head())


Base: (2397, 189)
Vegetation/fuel features: (2397, 9)
Merged: (2397, 197)
Saved: c:\Users\chaud\Desktop\repositories\wildfire-severity-v2\data\processed\calfire_with_gridmet_terrain_drought_veg.csv


,gridmet_id,Name,AcresBurned,severity_tier,fbfm40_code,fbfm40_group,evc_code,evh_code,canopy_cover
0,fire_00000,Creek Fire,65.0,Small,183.0,timber_litter,14.0,14.0,45.0
1,fire_00001,Taglio Fire,30.0,Small,93.0,nonburnable_or_other,64.0,64.0,0.0
2,fire_00002,Tulloch Fire,85.0,Small,102.0,grass,351.0,304.0,0.0
3,fire_00003,Metz Fire,3876.0,Large,102.0,grass,342.0,303.0,0.0
4,fire_00004,Wheatland Fire,156.0,Medium,122.0,grass_shrub,224.0,215.0,0.0


## 14. Quality checks

In [15]:
check = pd.read_csv(OUTPUT_PATH)

veg_cols = [
    c for c in [
        "fbfm40_code",
        "fbfm40_group",
        "evt_code",
        "evc_code",
        "evh_code",
        "canopy_cover",
        "fvt_code",
        "fvc_code",
    ]
    if c in check.columns
]

print("Vegetation/fuel columns:")
print(veg_cols)

print("\nMissingness:")
display(check[veg_cols].isna().mean().sort_values(ascending=False))

if "fbfm40_group" in check.columns:
    print("\nFBFM40 group distribution:")
    display(check["fbfm40_group"].value_counts(dropna=False))

if "severity_tier" in check.columns and "fbfm40_group" in check.columns:
    print("\nSeverity tier by FBFM40 group, normalized by row:")
    display(pd.crosstab(
        check["severity_tier"],
        check["fbfm40_group"],
        normalize="index"
    ).round(3))

if "severity_tier" in check.columns and "canopy_cover" in check.columns:
    print("\nMedian canopy cover by severity tier:")
    display(check.groupby("severity_tier")["canopy_cover"].median().to_frame("median_canopy_cover"))


Vegetation/fuel columns:
['fbfm40_code', 'fbfm40_group', 'evt_code', 'evc_code', 'evh_code', 'canopy_cover', 'fvt_code', 'fvc_code']

Missingness:


evc_code        0.003755
fbfm40_code     0.000000
fbfm40_group    0.000000
evt_code        0.000000
evh_code        0.000000
canopy_cover    0.000000
fvt_code        0.000000
fvc_code        0.000000
dtype: float64


FBFM40 group distribution:


fbfm40_group
grass                   731
grass_shrub             588
nonburnable_or_other    442
shrub                   333
timber_understory       164
timber_litter           139
Name: count, dtype: int64


Severity tier by FBFM40 group, normalized by row:


fbfm40_group,grass,grass_shrub,nonburnable_or_other,shrub,timber_litter,timber_understory
severity_tier,,,,,,
Extreme,0.253,0.217,0.108,0.181,0.114,0.127
Large,0.335,0.240,0.095,0.160,0.080,0.090
Medium,0.339,0.254,0.178,0.149,0.038,0.043
Small,0.288,0.245,0.211,0.125,0.059,0.072



Median canopy cover by severity tier:


,median_canopy_cover
severity_tier,
Extreme,0.0
Large,0.0
Medium,0.0
Small,0.0


## 15. Final output

If this notebook works, your vegetation/fuel-enriched modeling table is:

```text
data/processed/calfire_with_gridmet_terrain_drought_veg.csv
```

Recommended next step:

```text
08_enriched_eda.ipynb
```

Recommended README wording:

```text
Fuel and vegetation features were sampled at each wildfire ignition coordinate using LANDFIRE layers, including FBFM40 fuel model, existing vegetation type/cover/height, canopy cover, and fuel vegetation type/cover. Raw high-cardinality vegetation codes were retained for EDA, while FBFM40 was collapsed into broader fuel groups for interpretable modeling.
```
